In [ ]:
using QuantumOptics
using PyPlot

In [1]:
# ------------------------------
# Parameters normalized for photons per cavity lifetime
# ------------------------------
N = 70                   # Fock state size
λ = 2.0        # pump strength ratio, defines drive strength which is actual pump with respect to decay (1 is threshold)
b = 0.1                    # Injected coherent bias amplitude
κ = 1.0             # Signal decay rate (normalized to 1)
g = 0.1                   # Quantum noise scaling factor (>=1)
t_final = 200.0            # Final simulation time
dt = 0.01                  # Time step for evolution
K = 0.0         #0.4#0.05 #Kerr factor I guess I dont fucking know 
Δ = 0

drive_strength = λ*2*κ

4.0

In [ ]:
# ------------------------------
# Define Hilbert spaces and operators
# ------------------------------
basis = FockBasis(N)        # Signal mode Hilbert space


# Annihilation operators for signal and idler
a = destroy(basis)
# ------------------------------
# Hamiltonian construction
# ------------------------------
# Parametric squeezing: H_squeeze = iχ (a_s^† a_i^† - a_s a_i) pump hamiltonian
H_squeeze = 1im * (drive_strength/8) * (dagger(a)*dagger(a) - a*a)
#H_squeeze = (drive_strength/8) * (dagger(a)*dagger(a) + a*a)


# Injected coherent bias on signal: H_drive = ε (e^{iφ} a_s^† + e^{-iφ} a_s) bias hamiltonian
H_drive = 1im *b * (dagger(a)- a)
#H_drive = 1im*0.0 * (dagger(a) - a)

#H_kerr = (K/2)*(dagger(a)*dagger(a)*a*a) #DO K=0.5 FOR THIS TRUST ME kerr hamiltonian
#H_kerr = (K/2)*(dagger(a)*dagger(a)*a*a)
H_kerr = (K/2)*(dagger(a)*dagger(a)*a*a)

# Total Hamiltonian

H_detuning = Δ*dagger(a)*a
H = H_kerr + H_squeeze + H_drive +H_detuning

# ------------------------------
# Jump (collapse) operators with noise factor g. 
# ------------------------------

c_ops = [a , a^2]; #collapse operators
rates = [κ,g/2];  # scaling of the noise (scaling single photon noise by kappa)
#c_ops = [0.5 * a, 0.1/2 * a^2];


In [ ]:
# ------------------------------
# Initial state: vacuum in both modes
# ------------------------------

r = 0
theta = 0
z = r*exp(2*1im * theta) #squeezing parameter
SqueezingOperator = squeeze(basis,z) #squeezing operator for signal
#ψ0 = fockstate(basis, 0)
ψ0 = SqueezingOperator*fockstate(basis, 0)
ρ0 = dm(ψ0) #density matrix

# ------------------------------
# Time evolution parameters
# ------------------------------
times = 0:dt:t_final

# ------------------------------
# Perform master equation time evolution 
# ------------------------------
tout, ρt = timeevolution.master(times, ρ0, H, c_ops; rates=rates); 

In [ ]:
xvec = range(-10, 10, length=200)
yvec = range(-10, 10, length=200)
Winit = wigner(ρ0, xvec, yvec)'
figure()

imshow(Winit, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)),
       origin="lower", cmap="RdBu")
colorbar()
title("Initial Wigner Function")
xlabel("Re(α)"); ylabel("Im(α)")

In [ ]:
# ------------------------------
# Calculate photon number expectation values vs time
# ------------------------------
n = [real(expect(dagger(a)*a, ρ)) for ρ in ρt]
# Photon number vs time
figure()
plot(tout, n, label="Signal ⟨n⟩")
xlabel("Time")
ylabel("Photon number")
title("Photon Number Expectation vs Time")
grid(true)
#xlim(0,0.4)
tight_layout()
show()
println(length(tout))
println(tout[40])
#1001 = 10
#101 = 1
#40=0.4
println(n[end])

In [ ]:
# ------------------------------
# Reduced density matrix at final time
# ------------------------------
ρ_final = ρt[end]


using Printf
using LinearAlgebra: dot   # for the quadratic form v'Σv used to project variances

# ------------------------------
# Phase-space grids, Q and Wigner
# ------------------------------
xvec = range(-10, 10, length=1000)  # horizontal quadrature grid (x)
yvec = range(-10, 10, length=1000)  # vertical   quadrature grid (p)

# Husimi Q(α) = (1/π)⟨α|ρ|α⟩ on the (x,y) grid where α = (x + i y)/√2
Q = [real(expect(ρ_final, coherentstate(basis, (x+1im*y)/sqrt(2)))) / π for y in yvec, x in xvec]

# Wigner function W(x,y); transpose so indexing matches W[y,x] when plotting/images
W = wigner(ρ_final, xvec, yvec)'

# ------------------------------
# Peak of the Wigner lobe
# ------------------------------
max_ind = argmax(W)                # CartesianIndex(iy, ix) of the global maximum
iy, ix = Tuple(max_ind)
x_max = xvec[ix]                   # x-coordinate of the peak
y_max = yvec[iy]                   # y-coordinate of the peak

# Lobe direction and "amplitude" on that line (your r in the sketch)
θ_peak = atan(y_max, x_max)        # angle of the line that passes through the peak
r_peak = sqrt(x_max^2 + y_max^2)   # radius of the peak from the origin

# ------------------------------
# Lobe-aligned width from a local Wigner patch (no displacement)
# Idea: near the maximum, W(x,y) ~ local 2D Gaussian.  Estimate its
# covariance Σ from a small window around the peak, then project Σ on v
# (the unit vector along θ_peak) to get the 1-D σ on that line.
# ------------------------------
function lobe_width_from_wigner_patch(W, xvec, yvec, max_ind, θ_peak; halfwin::Int=6)
    yi, xi = Tuple(max_ind)   # integer indices of the peak (W is indexed [y,x])

    # Define a small window [yi±halfwin, xi±halfwin] clipped to grid bounds
    ys = max(1, yi-halfwin):min(length(yvec), yi+halfwin)
    xs = max(1, xi-halfwin):min(length(xvec), xi+halfwin)

    # Extract the patch and clamp tiny negatives (Wigner can have small negative values)
    Wpatch = @view W[ys, xs]
    w = max.(Array(Wpatch), 0.0)
    wsum = sum(w) + eps(Float64)  # normalization with numerical safety

    # Build coordinate matrices over the patch for weighted-moment calculations
    xv = xvec[collect(xs)]
    yv = yvec[collect(ys)]
    xpatch = repeat(reshape(xv, 1, :), length(yv), 1)
    ypatch = repeat(reshape(yv, :, 1), 1, length(xv))

    # Weighted centroid (μx, μy) of the local blob
    μx = sum(w .* xpatch) / wsum
    μy = sum(w .* ypatch) / wsum

    # Weighted covariance entries around the centroid
    dx = xpatch .- μx
    dy = ypatch .- μy
    cxx = sum(w .* (dx.^2)) / wsum
    cyy = sum(w .* (dy.^2)) / wsum
    cxy = sum(w .* (dx .* dy)) / wsum

    Σ = [cxx cxy; cxy cyy]                # local 2×2 covariance at the peak
    v = [cos(θ_peak), sin(θ_peak)]        # unit vector along the lobe direction

    # 1-D variance along θ_peak: σ^2 = v' Σ v
    σ = sqrt(max(dot(v, Σ*v), 0.0))

    # Convert standard deviation to Gaussian FWHM:  FWHM = 2√(2 ln 2) · σ ≈ 2.355σ
    FWHM = 2*sqrt(2*log(2)) * σ
    return σ, FWHM
end

# Compute local (lobe) width and SNR on that line
σ_lobe, FWHM_lobe = lobe_width_from_wigner_patch(W, xvec, yvec, max_ind, θ_peak; halfwin=6)
if !@isdefined ϵ
    ϵ = eps(Float64)            # define numeric safety epsilon if not defined earlier
end
SNR_r_over_FWHM = r_peak / (FWHM_lobe + ϵ)

println(@sprintf("LOBE (Wigner patch): σ=%.9f, FWHM=%.9f,  SNR r/FWHM = %.9f",
                 σ_lobe, FWHM_lobe, SNR_r_over_FWHM))

# ------------------------------
# Plotting: Q and W with the FWHM segment drawn along the lobe direction
# ------------------------------
figure(figsize=(16, 14))

# Q-function (top)
subplot(2, 1, 1)
imshow(Q, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)),
       origin="lower", cmap="viridis")
colorbar()
title(@sprintf("Q-function  (b=%.3g, g=%.3g)", abs(b), g))
xlabel("Re(α)"); ylabel("Im(α)")

# Wigner + one-sided FWHM segment from the peak along θ_peak (bottom)
subplot(2, 1, 2)
imshow(W, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)),
       origin="lower", cmap="RdBu")
colorbar()
title("Wigner Function")
xlabel("Re(α)"); ylabel("Im(α)")

# mark the peak
scatter([x_max], [y_max], color="black")

# draw the FWHM segment aligned with the lobe
x_end = x_max + FWHM_lobe * cos(θ_peak)
y_end = y_max + FWHM_lobe * sin(θ_peak)
plot([x_max, x_end], [y_max, y_end], "k-", linewidth=2)

tight_layout()
show()


In [ ]:
# Peak in Wigner (W is indexed [y, x])
max_ind = argmax(W)
iy, ix = Tuple(max_ind)
x_max = xvec[ix]
y_max = yvec[iy]

# Print the same items as in your picture
println(max_ind)                             # CartesianIndex(iy, ix)
println(x_max)                               # x at max
println(y_max)                               # y at max
println(W[max_ind])                          # Wigner value at max
println(atan(y_max, x_max)*180/pi)           # angle (deg)
r_peak = hypot(x_max, y_max)
println(r_peak)                              # radius

# Peak-based SNRs using your global σ and FWHM on that line
ϵ = eps(Float64)
SNR_peak_FWHM  = r_peak / (FWHM_lobe + ϵ)

@printf "Peak-based SNR  amplitude/FWHM≈ %.3f\n"  SNR_peak_FWHM
@printf "Angle is≈ %.3f\n"  atan(y_max, x_max)*180/pi

In [ ]:
figure()

imshow(W, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)),
       origin="lower", cmap="RdBu")

scatter(x_max,y_max, color = "black")
# FWHM line along the lobe direction (one-sided segment from the peak)
x_end = x_max + FWHM_lobe * cos(θ_peak)
y_end = y_max + FWHM_lobe * sin(θ_peak)
plot([x_max, x_end], [y_max, y_end], "k-", linewidth=2)

colorbar()
title("Wigner Function")
xlabel("Re(α)")
ylabel("Im(α)")

In [ ]:
p_b=sum(W[:,round(Int64,length(W[1,:])/2):end])/sum(W)
function prob_from_wigner(ρ; θ=0.0, xlim=10.0, ylim=10.0, grid=181)
    xvec = range(-xlim, xlim; length=grid)
    yvec = range(-ylim, ylim; length=grid)
    Wj = wigner(ρ, xvec, yvec)'

    Wc = max.(Wj, 0.0)
    dx = step(xvec); dy = step(yvec)
    Z  = sum(Wc) * dx * dy + eps(Float64)

    cθ, sθ = cos(θ), sin(θ)
    xv = repeat(reshape(collect(xvec), 1, :), length(yvec), 1)
    yv = repeat(reshape(collect(yvec), :, 1), 1, length(xvec))
    mask = (cθ .* xv .+ sθ .* yv) .>= 0.0

    p = (sum(Wc[mask]) * dx * dy) / Z
    return clamp(p, 0.0, 1.0)
end
prob_from_wigner(ρ_final; θ=0.0, xlim=10.0, ylim=10.0, grid=181)

In [ ]:
println(p_b)

In [ ]:
sum(W[:,1:round(Int64,length(W[1,:])/2)])/sum(W)

In [ ]:
function get_histogram_max(W, xvector,yvector)
    angles = zeros(length(W[1,:]) * length(W[:,1]))
    angular_content = zeros(length(angles))
    counter = 1
    max_W = maximum(W)

    for i in 1:length(W[1,:])
        for j in 1:length(W[:,1])
            angles[counter] = atan(yvector[i],xvector[j])
            angular_content[counter] = W[i,j]/max_W
            counter = counter+1
        end
    end

    y,x,_ = hist(angles*180/pi, weights = angular_content, bins = 51, density = true)
    #return x[argmax(y[1:ceil(Int, length(y)/2)])]
    return x[argmax(y[1:ceil(Int, length(y)/2)])]

end

In [ ]:
println(get_histogram_max(W,xvec,yvec))

In [ ]:
println(length(ρt))
#time_index_list = [1,100,250,300,500,1000] #make sure the length of this is even for the plotting process
time_index_list = [1,10] #make sure the length of this is even for the plotting process
#time_index_list = [1,7,8,9,10,20] #H_squeeze =  (λ/2) * (dagger(a)*dagger(a) -  a*a)

xvec = range(-10, 10, length=200)
yvec = range(-10, 10, length=200)
m_plot = length(time_index_list)
figure(figsize=(16, 5))

W_init = wigner(ρt[1], xvec, yvec)'
Q_init = [abs(expect(ρt[1], coherentstate(basis, (x+1im*y)/sqrt(2)))).^2 / π for y in yvec, x in xvec]
subplot(1,3,1) #signal plot
imshow(W_init, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)), origin="lower", cmap="RdBu")


time_ind = 20000
ρ_sample = ρt[time_ind]
Q_sample = [abs(expect(ρ_sample, coherentstate(basis, (x+1im*y)/sqrt(2)))).^2 / π for y in yvec, x in xvec]
W_sample = wigner(ρ_sample, xvec, yvec)'

subplot(1,3,2) #signal plot
imshow(W_sample, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)), origin="lower", cmap="RdBu")
title("W")
#xlabel("Re(α)")
#ylabel("Im(α)")


subplot(1,3,3) #signal plot
plot(yvec, W_sample[:,100]/maximum(W_sample[:,100]),color = "red", label="Signal ⟨n⟩") #final state on y axis 
plot(yvec, W_sample[100,:]/maximum(W_sample[100,:]),color = "blue", label="Signal ⟨n⟩")#final state on x axis
plot(yvec, W_init[100,:]/maximum(W_init[100,:]),color = "green", label="Signal ⟨n⟩")
    


tight_layout()



In [ ]:
time_index_list = [10,100,250,300,500,20001] #make sure the length of this is even for the plotting process
#time_index_list = [1,1000] #make sure the length of this is even for the plotting process
xvec = range(-10, 10, length=200)
yvec = range(-10, 10, length=200)
m_plot = length(time_index_list)
figure(figsize=(16, 14))
for j in 1:length(time_index_list)
    time_ind = time_index_list[j]
    ρ_sample = ρt[time_ind]
    Q_sample = [abs(expect(ρ_sample, coherentstate(basis, (x+1im*y)/sqrt(2)))).^2 / π for y in yvec, x in xvec]
    W_sample = wigner(ρ_sample, xvec, yvec)'
    subplot(2,m_plot,j) #signal plot
    imshow(Q_sample, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)), origin="lower", cmap="viridis")
    #colorbar()
    title("Q, t=$(tout[time_ind])")
    subplot(2,m_plot,j+m_plot) #signal plot
    imshow(W_sample, extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)), origin="lower", cmap="RdBu")
    title("W")
    #xlabel("Re(α)")
    #ylabel("Im(α)")

end

In [ ]:
using PyPlot
using PyCall

mpl_colors = pyimport("matplotlib.colors")
norm = mpl_colors.PowerNorm(gamma=0.5)   # gamma < 1 => more contrast in low/mid values

figure()
# Wigner function W(x,y); transpose so indexing matches W[y,x] when plotting/images
xvec = range(-10, 10, length=200)  # horizontal quadrature grid (x)
yvec = range(-10, 10, length=200)  # vertical   quadrature grid (p)

W = wigner(ρt[50], xvec, yvec)'
max_ind = argmax(W)
iy, ix = Tuple(max_ind)
x_max = xvec[ix]
y_max = yvec[iy]

imshow(W,
       extent=(minimum(xvec), maximum(xvec), minimum(yvec), maximum(yvec)),
       origin="lower",
       cmap="inferno",
       norm=norm)

#scatter(x_max,y_max, color = "yellow")
# FWHM line along the lobe direction (one-sided segment from the peak)
#x_end = x_max + FWHM_lobe * cos(θ_peak)
#y_end = y_max + FWHM_lobe * sin(θ_peak)
#plot([x_max, x_end], [y_max, y_end], "k-", linewidth=2)
#scatter(0,0,color = "black")
#scatter(0,0,color = "black")
# mark the origin
#scatter([0], [0]; color="white", s=60, marker="+")
# get axis limits
xmin, xmax = minimum(xvec), maximum(xvec)
ymin, ymax = minimum(yvec), maximum(yvec)

# horizontal line at y = 0
plot([xmin, xmax], [0, 0]; color="white", linestyle="--", linewidth=1)

# vertical line at x = 0
plot([0, 0], [ymin, ymax]; color="white", linestyle="--", linewidth=1)
#colorbar()
title("Wigner Function")
#xlabel("Re(α)")
#ylabel("Im(α)")

#savefig("biasSqueezed_state.svg", bbox_inches="tight", dpi=300)

In [ ]:
#size(ρt)

In [ ]:
tout